# Land-Use Classification — Training Notebook

Trains a multi-class U-Net segmentation model on the DeepGlobe Land Cover dataset (7 classes: urban_land, agriculture_land, rangeland, forest_land, water, barren_land, unknown).

**BEFORE RUNNING:**
1. Click **Add Input** (right panel) → attach your `cadastral-landuse-data` dataset.
2. In notebook Settings (right panel), set **Accelerator = GPU T4 x2** (or P100).
3. Run all cells top to bottom.


In [ ]:
import os
import glob
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices('GPU'))

## Configuration — edit paths here if your dataset names differ

In [ ]:
# EDIT THESE if your Kaggle dataset slug names are different
DATA_ROOT = "/kaggle/input/datasets/mageshkumarnagappan/cadastral-landuse-data/landuse"
PRETRAINED_CANDIDATES = []  # no land-use-specific pretrained checkpoint - always trains fresh

IMG_SIZE = 512
BATCH_SIZE = 8          # lower this to 4 if you hit out-of-memory errors
EPOCHS = 30
LEARNING_RATE = 1e-4

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Found candidate pretrained checkpoints:", PRETRAINED_CANDIDATES)
print("Data root:", DATA_ROOT, "exists:", os.path.exists(DATA_ROOT))
NUM_CLASSES = 7  # urban_land, agriculture_land, rangeland, forest_land, water, barren_land, unknown
CLASS_NAMES = ["urban_land", "agriculture_land", "rangeland", "forest_land", "water", "barren_land", "unknown"]


## Data pipeline
Reads matching filenames from `images/` and `masks/` subfolders for each split.

In [ ]:
def load_pair(img_path, mask_path):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    # IMPORTANT: land-use masks store CLASS INDICES (0-6), not a 0/255 binary
    # mask - use nearest-neighbor resize and do NOT divide by 255, or class
    # indices would be corrupted into meaningless fractional values.
    mask = tf.image.resize(mask, [IMG_SIZE, IMG_SIZE], method="nearest")
    mask = tf.cast(mask, tf.int32)
    return img, mask


def make_dataset(split, batch_size=BATCH_SIZE, shuffle=True, augment=False):
    images_dir = os.path.join(DATA_ROOT, split, "images")
    masks_dir = os.path.join(DATA_ROOT, split, "masks")

    image_paths = sorted(glob.glob(os.path.join(images_dir, "*.png")))
    mask_paths = [os.path.join(masks_dir, os.path.basename(p)) for p in image_paths]

    pairs = [(i, m) for i, m in zip(image_paths, mask_paths) if os.path.exists(m)]
    print(f"{split}: {len(pairs)} image/mask pairs found")
    if len(pairs) == 0:
        raise RuntimeError(f"No pairs found for split '{split}' - check DATA_ROOT path.")

    image_paths, mask_paths = zip(*pairs)
    ds = tf.data.Dataset.from_tensor_slices((list(image_paths), list(mask_paths)))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(2000, len(pairs)), reshuffle_each_iteration=True)
    ds = ds.map(load_pair, num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        def aug(img, mask):
            if tf.random.uniform([]) > 0.5:
                img = tf.image.flip_left_right(img)
                mask = tf.image.flip_left_right(mask)
            if tf.random.uniform([]) > 0.5:
                img = tf.image.flip_up_down(img)
                mask = tf.image.flip_up_down(mask)
            return img, mask
        ds = ds.map(aug, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_dataset("train", shuffle=True, augment=True)
val_ds = make_dataset("val", shuffle=False, augment=False)
test_ds = make_dataset("test", shuffle=False, augment=False)


## Quick visual sanity check before training

In [ ]:
for imgs, masks in train_ds.take(1):
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for i in range(4):
        axes[0, i].imshow(imgs[i].numpy())
        axes[0, i].set_title("image")
        axes[0, i].axis("off")
        axes[1, i].imshow(masks[i].numpy().squeeze(), cmap="tab10", vmin=0, vmax=NUM_CLASSES-1)
        axes[1, i].set_title("mask (class index)")
        axes[1, i].axis("off")
    plt.tight_layout()
    plt.show()


## Losses & metrics
Combined BCE + Dice loss handles the building/background imbalance (~85/15 split found in the data report) better than plain cross-entropy alone.

In [ ]:
def sparse_dice_loss(y_true, y_pred, smooth=1e-6):
    # y_true: (B,H,W,1) int32 class indices, y_pred: (B,H,W,NUM_CLASSES) softmax probs
    y_true = tf.cast(tf.squeeze(y_true, axis=-1), tf.int32)
    y_true_oh = tf.one_hot(y_true, NUM_CLASSES)
    intersection = tf.reduce_sum(y_true_oh * y_pred, axis=[0, 1, 2])
    union = tf.reduce_sum(y_true_oh, axis=[0, 1, 2]) + tf.reduce_sum(y_pred, axis=[0, 1, 2])
    dice_per_class = (2.0 * intersection + smooth) / (union + smooth)
    return 1.0 - tf.reduce_mean(dice_per_class)


def ce_dice_loss(y_true, y_pred):
    ce = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    return tf.reduce_mean(ce) + sparse_dice_loss(y_true, y_pred)


def mean_iou_metric(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(tf.squeeze(y_true, axis=-1), tf.int32)
    y_pred_labels = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
    ious = []
    for c in range(NUM_CLASSES):
        true_c = tf.cast(tf.equal(y_true, c), tf.float32)
        pred_c = tf.cast(tf.equal(y_pred_labels, c), tf.float32)
        intersection = tf.reduce_sum(true_c * pred_c)
        union = tf.reduce_sum(true_c) + tf.reduce_sum(pred_c) - intersection
        ious.append((intersection + smooth) / (union + smooth))
    return tf.reduce_mean(tf.stack(ious))


def pixel_accuracy(y_true, y_pred):
    y_true = tf.cast(tf.squeeze(y_true, axis=-1), tf.int32)
    y_pred_labels = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
    correct = tf.cast(tf.equal(y_true, y_pred_labels), tf.float32)
    return tf.reduce_mean(correct)


# aliases so the rest of the notebook (which refers to these names) still works
bce_dice_loss = ce_dice_loss
iou_metric = mean_iou_metric
dice_coef = pixel_accuracy  # repurposed as pixel accuracy for multi-class reporting


## Model
Builds a fresh U-Net from scratch, with a softmax output over NUM_CLASSES=7 channels (instead of a single sigmoid channel used for binary tasks).


In [ ]:
def build_unet_from_scratch(input_size=IMG_SIZE):
    inputs = tf.keras.Input((input_size, input_size, 3))

    def conv_block(x, filters):
        x = tf.keras.layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = tf.keras.layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        return x

    # Encoder
    c1 = conv_block(inputs, 32); p1 = tf.keras.layers.MaxPooling2D()(c1)
    c2 = conv_block(p1, 64);     p2 = tf.keras.layers.MaxPooling2D()(c2)
    c3 = conv_block(p2, 128);    p3 = tf.keras.layers.MaxPooling2D()(c3)
    c4 = conv_block(p3, 256);    p4 = tf.keras.layers.MaxPooling2D()(c4)

    # Bottleneck
    bn = conv_block(p4, 512)

    # Decoder
    u4 = tf.keras.layers.Conv2DTranspose(256, 2, strides=2, padding="same")(bn)
    u4 = tf.keras.layers.Concatenate()([u4, c4]); d4 = conv_block(u4, 256)
    u3 = tf.keras.layers.Conv2DTranspose(128, 2, strides=2, padding="same")(d4)
    u3 = tf.keras.layers.Concatenate()([u3, c3]); d3 = conv_block(u3, 128)
    u2 = tf.keras.layers.Conv2DTranspose(64, 2, strides=2, padding="same")(d3)
    u2 = tf.keras.layers.Concatenate()([u2, c2]); d2 = conv_block(u2, 64)
    u1 = tf.keras.layers.Conv2DTranspose(32, 2, strides=2, padding="same")(d2)
    u1 = tf.keras.layers.Concatenate()([u1, c1]); d1 = conv_block(u1, 32)

    outputs = tf.keras.layers.Conv2D(NUM_CLASSES, 1, activation="softmax")(d1)
    return tf.keras.Model(inputs, outputs, name="unet_from_scratch")


model = None
if len(PRETRAINED_CANDIDATES) > 0:
    checkpoint_path = PRETRAINED_CANDIDATES[0]
    print(f"Attempting to load pretrained checkpoint: {checkpoint_path}")
    try:
        model = tf.keras.models.load_model(checkpoint_path, compile=False)
        print("Loaded pretrained checkpoint successfully. Will fine-tune it.")
    except Exception as e:
        print(f"Could not load pretrained checkpoint ({e}). Falling back to fresh U-Net.")
        model = None

if model is None:
    print("No usable pretrained checkpoint found - building a fresh U-Net from scratch.")
    model = build_unet_from_scratch()

model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=bce_dice_loss,
    metrics=[iou_metric, dice_coef],
)

## Train

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, "landuse_model_best.keras"),
        save_best_only=True, monitor="val_loss", mode="min", verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=6, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

## Plot training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, key in zip(axes, ["loss", "iou_metric", "dice_coef"]):
    ax.plot(history.history[key], label="train")
    ax.plot(history.history["val_" + key], label="val")
    ax.set_title(key)
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=120)
plt.show()

## Evaluate on the held-out test set
These numbers go directly into your manuscript's Results table.

In [ ]:
test_results = model.evaluate(test_ds, return_dict=True)
print("Test set results:")
for k, v in test_results.items():
    print(f"  {k}: {v:.4f}")

## Visualize predictions on a few test samples

In [ ]:
for imgs, masks in test_ds.take(1):
    preds = model.predict(imgs)
    pred_labels = np.argmax(preds, axis=-1)
    n = min(4, imgs.shape[0])
    fig, axes = plt.subplots(3, n, figsize=(4 * n, 12))
    for i in range(n):
        axes[0, i].imshow(imgs[i].numpy()); axes[0, i].set_title("Image"); axes[0, i].axis("off")
        axes[1, i].imshow(masks[i].numpy().squeeze(), cmap="tab10", vmin=0, vmax=NUM_CLASSES-1); axes[1, i].set_title("Ground Truth"); axes[1, i].axis("off")
        axes[2, i].imshow(pred_labels[i], cmap="tab10", vmin=0, vmax=NUM_CLASSES-1); axes[2, i].set_title("Prediction"); axes[2, i].axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "sample_predictions.png"), dpi=120)
    plt.show()


## Save the final model
Download `landuse_model_final.keras` from the Output panel (right side) once this finishes — this is the file you'll use later in the fusion step.

In [ ]:
final_path = os.path.join(OUTPUT_DIR, "landuse_model_final.keras")
model.save(final_path)
print(f"Saved final model to: {final_path}")
print("Go to the 'Output' tab in the right-hand panel of this notebook to download it.")